# Run the experiments on Google Colab

This notebook reproduces all modelling experiments of the project on a Colab GPU by running `src/train.py`.

**Setup**
1. Select a GPU runtime: `Runtime → Change runtime type → GPU`.
2. Run the cells in order.

Results are written to Google Drive (`OUTPUT_DIR`). Every finished run is saved immediately, so if the runtime
disconnects, running the notebook again continues from the last finished run.

| Stage | Training data | Evaluation data | Runs |
|---|---|---|---|
| `test` | Jan–Aug (early stopping on Sep–Oct) | Nov–Dec, used once | 3 horizons × (2 baselines + XGBoost and LSTM × 5 feature sets × 3 seeds) |
| `walkforward` | months 1 … m | month m + 1 (May–Oct) | 3 horizons × 6 folds × (2 baselines + XGBoost and LSTM × 2 feature sets) |

## 1 · Settings

In [ ]:
SOURCE='github'          # 'github' to clone the repository, 'zip' to use a copy of the project stored in Google Drive
REPO_URL='https://github.com/sfeyzayildiz/mahmutbey-traffic-forecast.git'
ZIP_PATH='/content/drive/MyDrive/mahmutbey-traffic-forecast.zip'
OUTPUT_DIR='/content/drive/MyDrive/mahmutbey_results'

## 2 · Connect Google Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3 · Get the project

In [ ]:
import os
PROJECT='/content/mahmutbey-traffic-forecast'
!rm -rf $PROJECT
if SOURCE=='github':
    !git clone -q $REPO_URL $PROJECT
else:
    !unzip -q $ZIP_PATH -d /content/
os.chdir(PROJECT)
os.makedirs(OUTPUT_DIR, exist_ok=True)
!ls

## 4 · Install dependencies

In [ ]:
!pip install -q -r requirements.txt
import tensorflow as tf, xgboost
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
print('XGBoost', xgboost.__version__)

## 5 · Build the processed dataset
Runs the processing pipeline of notebook 02 and writes `data/processed/dataset.parquet`.

In [ ]:
import sys; sys.path.insert(0, '.')
from src.data import load_traffic, load_weather, load_fuel_events, ROOT
from src.processing import build_dataset
df, meta, _=build_dataset(load_traffic(), load_weather(), load_fuel_events())
(ROOT/'data'/'processed').mkdir(parents=True, exist_ok=True)
df.to_parquet(ROOT/'data'/'processed'/'dataset.parquet')
print(df.shape, meta)

## 6 · Smoke test
A short run (one seed, a few LSTM epochs) to check the environment. Results go to `results_quick.csv`
and are not used in the analysis.

In [ ]:
!python -m src.train --out $OUTPUT_DIR --quick

## 7 · Test stage
Each line of the log shows the run, its test MAE, the coverage of the 90% interval and the training time.

In [ ]:
!python -m src.train --out $OUTPUT_DIR --stage test

## 8 · Walk-forward stage

In [ ]:
!python -m src.train --out $OUTPUT_DIR --stage walkforward

## 9 · Summary and export
Shows the mean test results of the full models and writes `mahmutbey_results.zip`
(results table, predictions and MLflow database) next to `OUTPUT_DIR`.

In [ ]:
import shutil
from src.train import load_results
r=load_results(f'{OUTPUT_DIR}/results.csv')
print(f'{len(r)} runs finished')
display(r.query("stage=='test' and ablation=='full'")
         .groupby(['horizon', 'model'])[['MAE', 'RMSE', 'MAPE', 'R2', 'PICP90', 'MPIW90']].mean().round(2))
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f'Saved {OUTPUT_DIR}.zip')